# Lesson 10 — Poisson Regression: Counting & Offset

<h2 dir="rtl">פרק 1 — למה לא OLS על מנייה</h2>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import pickle as pkl
from scipy.stats import poisson

df = pkl.load(open('pkl/df_poisson_1.pkl', 'rb'))
df.head()

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(df['expirience'], df['accidents'], alpha=0.6)
plt.xlabel('Experience (months)', fontsize=18)
plt.ylabel('Accidents', fontsize=18)
plt.show()

In [ ]:
ols = sm.OLS(df['accidents'], df[['intercept', 'expirience']]).fit()
print(ols.summary())
print('\nPredicted accidents at experience = 0:', round(ols.predict(np.array([1, 0]))[0], 2))

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(df['expirience'], df['accidents'], alpha=0.6)
plt.plot(df['expirience'], ols.fittedvalues, color='orange', lw=3, label='OLS fit')
plt.axhline(0, color='red', ls='--', lw=1)
plt.xlabel('Experience (months)', fontsize=18)
plt.ylabel('Accidents', fontsize=18)
plt.legend(fontsize=14)
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(df['expirience'], ols.resid, color='red', alpha=0.6)
plt.axhline(0, color='black', lw=1)
plt.xlabel('Experience (months)', fontsize=18)
plt.ylabel('OLS residuals', fontsize=18)
plt.show()

<h2 dir="rtl">פרק 2 — התפלגות פואסון</h2>

In [ ]:
x = np.arange(0, 30)
plt.figure(figsize=(9, 6))
for lam in [2, 6, 12]:
    plt.plot(x, poisson.pmf(x, lam), 'o-', label=f'$\\lambda$ = {lam}')
plt.xlabel('y  (count)', fontsize=18)
plt.ylabel('P(Y = y)', fontsize=18)
plt.legend(fontsize=14)
plt.show()

<h2 dir="rtl">פרק 3 — log-link ורגרסיה פואסונית</h2>

In [ ]:
pois = sm.GLM(df['accidents'], df[['intercept', 'expirience']],
              family=sm.families.Poisson()).fit()
print(pois.summary())

In [ ]:
o = np.argsort(df['expirience'].values)
plt.figure(figsize=(8, 6))
plt.scatter(df['expirience'], df['accidents'], alpha=0.6)
plt.plot(df['expirience'].values[o], pois.fittedvalues.values[o], color='black', lw=3, label='Poisson fit')
plt.xlabel('Experience (months)', fontsize=18)
plt.ylabel('Accidents', fontsize=18)
plt.legend(fontsize=14)
plt.show()

<h2 dir="rtl">פרק 4 — בדיקת mean = variance</h2>

In [ ]:
edges = np.linspace(df['expirience'].min(), df['expirience'].max(), 11)
means, variances = [], []
for lo, hi in zip(edges[:-1], edges[1:]):
    bucket = df.loc[(df['expirience'] >= lo) & (df['expirience'] < hi), 'accidents']
    if len(bucket) > 10:
        means.append(bucket.mean())
        variances.append(bucket.var())

plt.figure(figsize=(8, 6))
plt.scatter(means, variances, s=60)
lim = max(max(means), max(variances))
plt.plot([0, lim], [0, lim], '--', color='orange', label='variance = mean')
plt.xlabel('Bucket mean', fontsize=18)
plt.ylabel('Bucket variance', fontsize=18)
plt.legend(fontsize=14)
plt.show()

<h2 dir="rtl">פרק 5 — יחס הקצב (IRR)</h2>

In [ ]:
b1 = pois.params['expirience']
print('b1       =', round(b1, 4))
print('IRR = e^b1 =', round(np.exp(b1), 4))

# confirm numerically: predicted rate at x and x+1
p5 = pois.predict(np.array([1, 5]))[0]
p6 = pois.predict(np.array([1, 6]))[0]
print(f'\npredict(exp=5) = {p5:.4f}')
print(f'predict(exp=6) = {p6:.4f}')
print(f'ratio          = {p6 / p5:.4f}')

<h2 dir="rtl">פרק 6 — חשיפה לסיכון</h2>

In [ ]:
data = pkl.load(open('pkl/data_poisson_2.pkl', 'rb'))
data[['Segment', 'accidents', 'Miles', 'AADT']].head()

In [ ]:
plt.figure(figsize=(8, 6))
plt.hist(data['Miles'], bins=60)
plt.xlabel('Segment length (miles)', fontsize=18)
plt.ylabel('Count of segments', fontsize=18)
plt.show()

<h2 dir="rtl">פרק 7 — offset במודל</h2>

In [ ]:
no_offset = sm.GLM(data['accidents'], data[['intercept', 'AADT']],
                   family=sm.families.Poisson()).fit()

with_offset = sm.GLM(data['accidents'], data[['intercept', 'AADT']],
                     family=sm.families.Poisson(),
                     offset=np.log(data['Miles'])).fit()

print('Without offset:  b0 = %.4f,  b1(AADT) = %.6f' % (no_offset.params['intercept'], no_offset.params['AADT']))
print('With offset:     b0 = %.4f,  b1(AADT) = %.6f' % (with_offset.params['intercept'], with_offset.params['AADT']))

In [ ]:
print(with_offset.summary())

<h2 dir="rtl">פרק 8 — חיזוי עם offset + סיכום</h2>

In [ ]:
AADT = 400
m = 3   # segment length in miles

rate = with_offset.predict(np.array([1, AADT]))[0]   # accidents per mile
expected = m * rate
print(f'Rate (per mile) at AADT={AADT}: {rate:.4f}')
print(f'Expected accidents on a {m}-mile segment: {expected:.4f}')

### Write your answer here.

<div dir="rtl">

תרגיל אופציונלי לבית: נתונים על מספר תקלות במכונות לפי גיל המכונה, כאשר לכל מכונה מספר שעות הפעלה שונה:

- הריצו רגרסיה פואסונית עם offset מתאים.
- מה ה-IRR לכל שנת גיל נוספת? פרשו אותו.

</div>